In [1]:
import tensorflow as tf
from config import *
from tensorflow.keras import Sequential
from tensorflow.keras import layers,models
import matplotlib.pyplot as plt
import laspy
import numpy as np

2026-03-03 18:12:49.670921: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 18:12:49.864811: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-03 18:12:50.898265: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


model = models.Sequential()

model.add(layers.Conv3D(
    filters = 16,
    kernel_size = (3,3,3),
    activation = 'relu',
    input_shape = (32,32,32,1)
))
model.add(layers.MaxPooling3D(pool_size=(2,2,2)))
model.add(layers.GlobalAveragePooling3D())
model.add(layers.Dense(10,activation='softmax'))


model.summary()

In [2]:
data_input = '/home/ajai-krishna/work/GEO_AI/data/outputs/ground_points.tif'

In [ ]:

model = models.Sequential()

model.add(layers.Conv2D(
    filters=16,
    kernel_size=(3,3),
    activation='relu',
    input_shape=(32,32,1)
))

model.add(layers.MaxPooling2D(pool_size=(2,2)))

model.add(layers.GlobalAveragePooling2D())

model.add(layers.Dense(20, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [4]:
las = laspy.read("/home/ajai-krishna/work/GEO_AI/data/input/laz/DEVDI_POINT CLOUD (511671).las")

In [5]:
x = las.x
y = las.y
z = las.z

In [6]:
# Normalize coordinates
x_norm = (x - x.min()) / (x.max() - x.min())
y_norm = (y - y.min()) / (y.max() - y.min())
z_norm = (z - z.min()) / (z.max() - z.min())

grid_size = 32

voxel = np.zeros((grid_size, grid_size, grid_size))

xi = (x_norm * (grid_size-1)).astype(int)
yi = (y_norm * (grid_size-1)).astype(int)
zi = (z_norm * (grid_size-1)).astype(int)

voxel[zi, yi, xi] = 1

In [7]:
X = voxel.reshape(1, 32, 32, 32, 1)
y = np.array([[1,0,0,0,0,0,0,0,0,0]])  # 10-class example
history = model.fit(X,y,epochs=50)

Epoch 1/50


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 with name 'None' of layer 'conv2d' is incompatible with the layer: expected axis -1 of input shape to have value 1, but received input with shape (None, 32, 32, 32)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None, 32, 32, 32, 1), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [ ]:
data = "/home/ajai-krishna/work/GEO_AI/data/input/laz/DEVDI_POINT CLOUD (511671).las"
dataset = tf.data.Dataset.from_tensor_slices(data)
dataset = dataset.batch(32)
history = model.fit(dataset, epochs=10) 
plt.plot(history.history['loss'])   
plt.ylabel('loss')    